In [0]:
from pyspark.sql.functions import sum, avg, round

# monthly sales trend
monthly_sales = spark.read.table("ecommerce.e_comm_gold.factSales")
monthly_sales.createOrReplaceTempView("overall_sales")

# calculate metrics 
agg_sales_metrics = spark.sql("""        
        select 
            count(order_id) as total_orders
            , sum(order_quantity) as total_items_sold
            , round(sum(sale_amount),2) as total_revenue
            , round(avg(sale_amount),2) as avg_order_value
            , round(avg(order_quantity),2) as avg_items_per_order
            , current_timestamp() as load_ts
        from overall_sales
        """)
    
# write to delta table

agg_sales_metrics.write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("ecommerce.e_comm_gold.fact_agg_sales_metrics")